# SARIMA MODEL: NBA API

`Authors:` Sarah Beltran, Sofia Maldonado & Aissa 

`Date:` 17/02/2026

---

In [ ]:
# %pip install nbformat nba_api

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.30.0-cp311-cp311-win_amd64.whl.metadata (4.2 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
Using cached rpds_py-0.30.0-cp311-cp311-win_amd64.whl (236 kB)

   ---------------------------------------- 0/7 [fastjsonschema]
   ---------------------------------------- 0/7 [fastjsonschema]
   ---------------------------------------- 0/7 [fastjsonschema]
   -----------------------

In [3]:
# libraries 
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd
import plotly.graph_objects as go
import nbformat

In [5]:
games = leaguegamefinder.LeagueGameFinder(
    season_type_nullable='Regular Season'
).get_data_frames()[0]
df_games = pd.DataFrame(games)

# converts date
df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])

# regular season 2024-2025
# SEASON_ID: '22024'
df_games = df_games[df_games['SEASON_ID'] == '22024']

# no play-in/offs
df_clean = df_games[df_games['GAME_DATE'] <= '2025-02-15'].copy()

# ended games
df_clean = df_clean[df_clean['WL'].notna()]

# total points per game
df_clean['total_points'] = df_clean['PTS']

# daily time series points per day
ts_nba = (
    df_clean
    .groupby('GAME_DATE')['total_points']
    .sum()
    .asfreq('D')
    .fillna(0)
)

In [6]:
# original time series graph
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=ts_nba.index,
        y=ts_nba.values,
        mode='lines',
        name='Puntos Diarios'
    )
)

fig.update_layout(
    title='Volumen Diario de Puntos en la NBA (Regular Season)',
    xaxis_title='Fecha',
    yaxis_title='Total de Puntos'
)

fig.show()